In [ ]:
import numpy as np
import scipy.signal
import time
from typing import Optional, Tuple
import os
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
from pathlib import Path

path = os.getcwd()+ "/data/"

def ensure_n_lead_time_channel(x: np.ndarray):
    if x.ndim == 3:
        x = x[..., np.newaxis]
    elif x.ndim == 4:
        pass
    else:
        raise ValueError(f"Expected 3D or 4D array, got shape={x.shape}")

    if x.shape[1] != 12:
        raise ValueError(f"Expected 12 leads at axis=1, got shape={x.shape}")

    if x.shape[3] != 1:
        raise ValueError(f"Expected channel dim=1 at axis=3, got shape={x.shape}")

    return x


def baseline_wander_removal_single(ecg_12xt: np.ndarray, fs: int = 250):
    if ecg_12xt.ndim != 2 or ecg_12xt.shape[0] != 12:
        raise ValueError(f"Expected (12, T), got {ecg_12xt.shape}")

    win1 = int(np.round(0.2 * fs)) + 1
    win2 = int(np.round(0.6 * fs)) + 1

    if win1 % 2 == 0: win1 += 1
    if win2 % 2 == 0: win2 += 1

    out = np.empty_like(ecg_12xt)
    for lead in range(12):
        baseline = scipy.signal.medfilt(ecg_12xt[lead, :], kernel_size=win1)
        baseline = scipy.signal.medfilt(baseline, kernel_size=win2)
        out[lead, :] = ecg_12xt[lead, :] - baseline
    return out


def compute_per_lead_percentile_bounds(
    x_n12t1: np.ndarray,
    lower_q: float = 0.1,
    upper_q: float = 99.9
) :
    x_n12t1 = ensure_n_lead_time_channel(x_n12t1)

    lower = np.empty(12, dtype=np.float64)
    upper = np.empty(12, dtype=np.float64)
    for lead in range(12):
        vals = x_n12t1[:, lead, :, 0].reshape(-1)
        lower[lead] = np.nanpercentile(vals, lower_q)
        upper[lead] = np.nanpercentile(vals, upper_q)
    return lower, upper


def winsorize_per_lead(x_n12t1: np.ndarray, lower: np.ndarray, upper: np.ndarray):
    x_n12t1 = ensure_n_lead_time_channel(x_n12t1)
    if lower.shape != (12,) or upper.shape != (12,):
        raise ValueError("lower/upper must be shape (12,)")

    y = x_n12t1.copy()
    for lead in range(12):
        y[:, lead, :, 0] = np.clip(y[:, lead, :, 0], lower[lead], upper[lead])
    return y


def normalize_per_lead(x_n12t1: np.ndarray, mean_12: np.ndarray, std_12: np.ndarray):
    x_n12t1 = ensure_n_lead_time_channel(x_n12t1)
    mean_12 = np.asarray(mean_12, dtype=np.float64)
    std_12  = np.asarray(std_12, dtype=np.float64)
    if mean_12.shape != (12,) or std_12.shape != (12,):
        raise ValueError("mean_12/std_12 must be shape (12,)")

    y = x_n12t1.astype(np.float32, copy=True)
    for lead in range(12):
        y[:, lead, :, 0] = (y[:, lead, :, 0] - mean_12[lead]) / std_12[lead]
    return y


def baseline_joblib(x_n12t1, fs=250, n_jobs=None, batch_size=64):
    X = ensure_n_lead_time_channel(x_n12t1)
    N, _, T, _ = X.shape
    if n_jobs is None:
        n_jobs = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

    results = Parallel(n_jobs=n_jobs)(
        delayed(baseline_wander_removal_single)(X[i, :, :, 0], fs) for i in tqdm(range(N))
    )
    return np.asarray(results, dtype=X.dtype)

def preprocess_ecg_array(
    x_n12t: np.ndarray,
    fs: int = 250,
    do_baseline_removal: bool = True,
    do_winsorize: bool = True,
    downsample_to_2500 = True,
    lower_q: float = 0.1,
    upper_q: float = 99.9,
    train_mean_12: Optional[np.ndarray] = None,
    train_std_12: Optional[np.ndarray] = None,
    batch_size: int = 256,
    return_intermediates: bool = False
):
    x = ensure_n_lead_time_channel(x_n12t)
    
    #  Downsample by taking every other sample
    if downsample_to_2500:
        x = x[:, :, ::2]  # (n,12,2500)

    inter = {}
    inter["x_input_n12t1"] = x

    if do_baseline_removal:
        x = baseline_joblib(x, fs=fs, n_jobs=64,batch_size=batch_size)
    inter["x_wander_removed_n12t1"] = x

    if do_winsorize:
        lower, upper = compute_per_lead_percentile_bounds(x, lower_q=lower_q, upper_q=upper_q)
        x = winsorize_per_lead(x, lower, upper)
        inter["winsor_lower_12"] = lower
        inter["winsor_upper_12"] = upper
    inter["x_winsorized_n12t1"] = x

    if train_mean_12 is None or train_std_12 is None:
        mean_12 = np.mean(x, axis=(0, 2, 3)).astype(np.float64)  # (12,)
        std_12  = np.std(x,  axis=(0, 2, 3)).astype(np.float64)  # (12,)
    else:
        mean_12 = np.asarray(train_mean_12, dtype=np.float64)
        std_12  = np.asarray(train_std_12, dtype=np.float64)

    x = normalize_per_lead(x, mean_12, std_12)
    inter["norm_mean_12"] = mean_12
    inter["norm_std_12"]  = std_12
    inter["x_normalized_n12t1"] = x
    X_model = x
    
    inter["X_model_n1t12"] = X_model

    return (X_model, inter) if return_intermediates else X_model

In [ ]:
x_tr = np.load(path + 'x_tr.npy',allow_pickle=True)
x_tst  = np.load(path + 'x_tst.npy',allow_pickle=True)
x_val = np.load(path + 'x_val.npy',allow_pickle=True)

X_tr_prep = preprocess_ecg_array(
     x_tr,
     fs=250,
     do_baseline_removal=True,
     do_winsorize=True,
     downsample_to_2500 = True,
     lower_q=0.1,
     upper_q=99.9,
     train_mean_12=None,
     train_std_12=None,
     batch_size=128,   
 )
np.save(path+"x_tr_prep.npy", X_tr_prep.astype(np.float32))

X_tst_prep = preprocess_ecg_array(
     x_tst,
     fs=250,
     do_baseline_removal=True,
     do_winsorize=True,
     downsample_to_2500 = True,
     lower_q=0.1,
     upper_q=99.9,
     train_mean_12=None,
     train_std_12=None,
     batch_size=128,  
 )
np.save(path+"x_tst_prep.npy", X_tst_prep.astype(np.float32))

X_val_prep = preprocess_ecg_array(
     x_val,
     fs=250,
     do_baseline_removal=True,
     do_winsorize=True,
     downsample_to_2500 = True,
     lower_q=0.1,
     upper_q=99.9,
     train_mean_12=None,
     train_std_12=None,
     batch_size=128,   
 )
np.save(path+"x_val_prep.npy", X_val_prep.astype(np.float32))


In [ ]:
x_tst = np.load(path+"x_tst.npy",allow_pickle=True)
X_model = np.load(path + 'x_tst_prep.npy',allow_pickle=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_ecg(sample, label,lead_names=None, fs=500):

    n_leads,timesteps = sample.shape
 
    if lead_names is None:
        lead_names = ["I","II","III","aVR","aVL","aVF",
                      "V1","V2","V3","V4","V5","V6"]

    time_axis = np.arange(timesteps) / fs

    plt.figure(figsize=(15, 12))

    if label is not None:
        plt.suptitle(f"ECG Sample - Label: {label}", fontsize=18, y=1.02)
    else:
        plt.suptitle("ECG Sample", fontsize=18, y=1.02)
        
    for i in range(n_leads):
        plt.subplot(6, 2, i+1)
        plt.plot(time_axis, sample[i,:], linewidth=0.8)
        plt.title(lead_names[i])
        plt.grid(True)
        plt.tight_layout()

    plt.show()

In [ ]:
print(x_tst.shape)
sample_id = 1000
sample = x_tst[sample_id,:,:,0]       
label  = "raw"   
plot_ecg(sample,label)

In [ ]:
sample2 = X_model[sample_id,:,:,0]      
label  = "after processing"   
plot_ecg(sample2,label)


In [ ]:
def ecg_full_summary(X, y, lead_axis=1):

    y = np.asarray(y).reshape(-1)

    X = np.asarray(X)
    nsample = X.shape[0]
    nlead = X.shape[lead_axis]
    ntime = X.shape[2] if lead_axis == 1 else X.shape[1]

    if lead_axis != 1:
        X = np.moveaxis(X, lead_axis, 1)  # now (nsample, lead, time)

    lead_summary = []
    for l in range(nlead):
        data = X[:, l, :].reshape(-1)

        lead_summary.append({
            "lead": l,
            "min": np.nanmin(data),
            "max": np.nanmax(data),
            "mean": np.nanmean(data),
            "std": np.nanstd(data),
            "contains_NaN": np.isnan(data).any()
        })

    lead_df = pd.DataFrame(lead_summary)

    var_matrix = np.nanvar(X, axis=2)

    var_summary = []
    for l in range(nlead):
        v = var_matrix[:, l]

        var_summary.append({
            "lead": l,
            "var_mean": np.nanmean(v),
            "var_std": np.nanstd(v),
            "var_min": np.nanmin(v),
            "var_max": np.nanmax(v),
            "contains_NaN": np.isnan(v).any()
        })

    var_df = pd.DataFrame(var_summary)

    label_unique, label_counts = np.unique(y, return_counts=True)
    label_percent = 100 * label_counts / len(y)

    label_df = pd.DataFrame({
        "label": label_unique,
        "count": label_counts,
        "percent": label_percent
    })

    print("\n========== ECG BASIC SUMMARY ==========")
    print(f"Samples     : {nsample}")
    print(f"Leads       : {nlead}")
    print(f"Time length : {ntime}")
    print("=======================================")
    print(lead_df)

    print("\n========== ECG VARIANCE SUMMARY ==========")
    print("Variance computed per sample along time axis")
    print("-------------------------------------------")
    print(var_df)

    print("\n========== LABEL SUMMARY =============")
    print(f"Total labels: {len(y)}")
    print(f"Contains NaN: {np.isnan(y).any()}")
    print("--------------------------------------")
    print(label_df)
    print("======================================\n")

    return lead_df, var_df, label_df


In [ ]:
x_tr_prep = np.load(path + 'x_tr_prep.npy',allow_pickle=True)
y_tr  = np.load(path + 'y_tr.npy',allow_pickle=True)
x_tr  = np.load(path + 'x_tr.npy',allow_pickle=True)

In [ ]:
lead_df, var_df, label_df = ecg_full_summary(x_tr, y_tr, lead_axis=1)

In [ ]:
lead_df, var_df, label_df = ecg_full_summary(x_tr_prep, y_tr, lead_axis=1)